In [2]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import svd
from PIL import Image
import matplotlib.patches as mpatches
import math
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit
from qiskit_aer.noise import (NoiseModel, depolarizing_error)
from matplotlib import pyplot as plt
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library.standard_gates import RYGate
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit_ibm_runtime.options.sampler_options import SamplerOptions
from qiskit.transpiler import generate_preset_pass_manager
from qiskit import transpile
import time
import json
from datetime import datetime
from matplotlib.ticker import FuncFormatter
import numpy as np
import matplotlib.pyplot as plt

In [3]:
def compress_count_keys(counts):
    """
    Changes the key values of the counts dictionary from bit strings to an int string for storage effiency 
    """
    compressed_counts = {}
    for key, value in counts.items():
        # reverse the key string
        key = key[::-1]
        # Remove leading zeros
        compressed_key = key.lstrip('0')
        # If the key becomes empty after stripping, it means it was all zeros
        if compressed_key == '':
            compressed_key = '0'
        # convert bit string into integer
        compressed_key = str(int(compressed_key, 2))
        compressed_counts[compressed_key] = value
    return compressed_counts


def decompress_count_keys(compressed_counts, total_qubits):
    """
    Reverses the compressing process of compress_count_keys when loading counts data
    """
    decompressed_counts = {}
    for key, value in compressed_counts.items():
        # convert integer key back to binary string
        bin_key = bin(int(key))[2:]  # remove '0b' prefix
        # Pad with leading zeros to match total_qubits
        bin_key = bin_key.zfill(total_qubits)
        # reverse the key string back to original order
        bin_key = bin_key[::-1]
        decompressed_counts[bin_key] = value
    return decompressed_counts


In [4]:
def clean_runtime_result(result):
    """Clean Qiskit Runtime result into human-friendly JSON-safe format."""
    cleaned = {"results": []}

    # Case 1: Dictionary result (common for runtime jobs retrieved by job_id)
    if isinstance(result, dict):
        for exp in result.get("results", []):
            cleaned_exp = {}
            if "quasi_dists" in exp:  # Sampler results
                cleaned_exp["quasi_dists"] = [
                    {str(k): float(v) for k, v in dist.items()}
                    for dist in exp["quasi_dists"]
                ]
            if "values" in exp:  # Estimator results
                cleaned_exp["values"] = [float(v) for v in exp["values"]]
            if "metadata" in exp:
                cleaned_exp["metadata"] = make_serializable(exp["metadata"])
            cleaned["results"].append(cleaned_exp)
        if "metadata" in result:
            cleaned["metadata"] = make_serializable(result["metadata"])

    # Case 2: Object with .results attribute (PrimitiveResult-like)
    elif hasattr(result, "results"):
        for exp in result.results:
            exp_data = {}
            if hasattr(exp, "data") and hasattr(exp.data, "get"):
                if "quasi_dists" in exp.data:  # Sampler
                    exp_data["quasi_dists"] = [
                        {str(k): float(v) for k, v in dist.items()}
                        for dist in exp.data["quasi_dists"]
                    ]
                if "values" in exp.data:  # Estimator
                    exp_data["values"] = [float(v) for v in exp.data["values"]]
            if hasattr(exp, "metadata"):
                exp_data["metadata"] = make_serializable(exp.metadata)
            cleaned["results"].append(exp_data)
        if hasattr(result, "metadata"):
            cleaned["metadata"] = make_serializable(result.metadata)

    # Case 3: Legacy qiskit.result.Result
    elif hasattr(result, "to_dict"):
        cleaned = make_serializable(result.to_dict())

    else:
        cleaned["raw"] = str(result)
    return cleaned


In [5]:
def save_job_results(job_id, service):
    json_jobid_fname = f"job_results_{job_id}.json"  # Chris's code
#    json_jobid_fname = f"job_submission_info_{job_id}.json"  # new code code
    # check if json_jobid_fname was created and return counts if so
    output_dir = "JobOutputs" 
    # Check if directory exists; if not, create it
    if not os.path.isdir(output_dir):
        os.makedirs(output_dir)
    job = service.job(job_id)
    # Fetch the results
    result = job.result()
    # Get and clean results
    raw_result = job.result()
    cleaned_result = clean_runtime_result(raw_result)
    # Job metadata
    creation_date = job.creation_date
    status_attr = job.status if not callable(job.status) else job.status()
    if hasattr(status_attr, "name"):  # Enum-like
        status_str = status_attr.name
    else:  # Already a string
        status_str = str(status_attr)
    counts = [result[i].data.c.get_counts()  for i in range(len(result))]
    compressed_count = []
    for count in counts:
        # compress count keys
        compressed_count.append(compress_count_keys(count))
    job_info = {
        "job_id": job.job_id(),
        "program_id": getattr(job, "program_id", None),
        "backend": job.backend().name if job.backend() else None,
        "creation_date": creation_date.isoformat() if isinstance(creation_date, datetime) else str(creation_date),
        "status": status_str
    }
    print("Counts dict of length:", len(counts))
    # Combine job info + results
    total_qubits = len(list(counts[0].keys())[0])
    final_export = {
        "counts":  compressed_count,
        # length of one key string in counts
        "total_qubits": total_qubits,
        "job_info": job_info,
        "results": cleaned_result #NOTE: This can take up a lot of memory depending on the job TODO: Figure out whats actually needed from the raw results and trim it down
    }
    # (Optional) Save dictionary to a JSON file
    with open(os.path.join(output_dir, json_jobid_fname), "w") as f:
        json.dump(final_export, f, indent=4)
    print(f"Universal cleaned job results saved to {json_jobid_fname}")
    return counts



In [12]:
#job_id = "d7v67a3ack5s73bf1kk0" 
#job_id = "d80fjdkinasc738v52u0" 

account_credentials_name = "LBNL QCAN Instance" # ewb
service = QiskitRuntimeService(name=account_credentials_name)
service.active_account() # this will print the active account information, including the private API key.

#counts= save_job_results(job_id, service)

management.get:WARNING:2026-05-11 18:06:51,714: Loading saved account: LBNL QCAN Instance
qiskit_runtime_service.__init__:WARNING:2026-05-11 18:06:56,574: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: m5004-eu, m5004-us. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


{'channel': 'ibm_quantum_platform',
 'url': 'https://cloud.ibm.com',
 'token': 'fX1Ozhbq1eazQ2Op67WL3-PMaGjYx-bIQlitYk_vHIk9',
 'verify': True,
 'private_endpoint': False}

In [7]:
# single job retrieval
#job_id = "d80fjdkinasc738v52u0" 
#counts= save_job_results(job_id, service)

Counts dict of length: 1
Universal cleaned job results saved to job_results_d80fjdkinasc738v52u0.json


In [13]:
#
# Plant Tile Test
# ibm marrakesh runs
# 5/11/2026
jobs = ["d816h7o0bvlc73d0m5vg", "d816h7o0bvlc73d0m600", "d816h86gbeec73ak5b50", "d816h8ftjchs73bmfmr0"]

for job_id in jobs:
    print(f"Attempting retrieval for Job id = {job_id}")
    counts= save_job_results(job_id, service)

Attempting retrieval for Job id = d816h7o0bvlc73d0m5vg
Counts dict of length: 64
Universal cleaned job results saved to job_results_d816h7o0bvlc73d0m5vg.json
Attempting retrieval for Job id = d816h7o0bvlc73d0m600
Counts dict of length: 16
Universal cleaned job results saved to job_results_d816h7o0bvlc73d0m600.json
Attempting retrieval for Job id = d816h86gbeec73ak5b50
Counts dict of length: 4
Universal cleaned job results saved to job_results_d816h86gbeec73ak5b50.json
Attempting retrieval for Job id = d816h8ftjchs73bmfmr0
Counts dict of length: 1
Universal cleaned job results saved to job_results_d816h8ftjchs73bmfmr0.json


In [16]:
#
# synthetic Tile Test
# ibm marrakesh runs
# 5/11/2026
jobs = ["d817bpntjchs73bmgm50", "d817bpo0bvlc73d0n5bg", "d817bpvoha1c73bk0peg",  "d817bq7tjchs73bmgm6g"]

for job_id in jobs:
    print(f"Attempting retrieval for Job id = {job_id}")
    counts= save_job_results(job_id, service)

Attempting retrieval for Job id = d817bpntjchs73bmgm50
Counts dict of length: 64
Universal cleaned job results saved to job_results_d817bpntjchs73bmgm50.json
Attempting retrieval for Job id = d817bpo0bvlc73d0n5bg
Counts dict of length: 16
Universal cleaned job results saved to job_results_d817bpo0bvlc73d0n5bg.json
Attempting retrieval for Job id = d817bpvoha1c73bk0peg
Counts dict of length: 4
Universal cleaned job results saved to job_results_d817bpvoha1c73bk0peg.json
Attempting retrieval for Job id = d817bq7tjchs73bmgm6g
Counts dict of length: 1
Universal cleaned job results saved to job_results_d817bq7tjchs73bmgm6g.json
